In [50]:
!pip install -q langchain langchain-community langchain-ollama
!pip install -q langchain-groq
import pandas as pd
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama

In [51]:
df = pd.read_csv("/content/realistic_autonomous_data_analyst_dataset.csv")

In [52]:
df.head()

,OrderID,OrderDate,Region,Category,Product,Quantity,UnitPrice,Revenue,Profit,CustomerAge,CustomerGender
0,ORD00001,2025-07-12,North,Electronics,Smartphone,1,26207.75,26207.75,2738.43,55,Female
1,ORD00002,2025-10-27,West,Electronics,Laptop,1,50530.65,50530.65,5218.10,32,Female
2,ORD00003,2024-08-08,East,Furniture,Chair,1,4732.93,4732.93,1009.97,24,Male
3,ORD00004,2025-04-15,North,Clothing,T-Shirt,2,874.62,1749.24,452.51,42,Male
4,ORD00005,2024-10-23,North,Furniture,Chair,6,4569.62,27417.72,4301.12,32,Male


In [53]:
summary = {
    # DATASET OVERVIEW
    "rows": len(df),
    "columns": df.columns.tolist(),

    # DATA TYPES
    "data_types": {
        col: str(dtype)
        for col, dtype in df.dtypes.items()
    },

    # DATA QUALITY
    "missing_values":
        df.isnull().sum().to_dict(),
    "duplicate_rows":
        int(df.duplicated().sum()),

    # DISTRIBUTIONS
    "region_distribution":
        df["Region"].value_counts().to_dict()
        if "Region" in df.columns else {},

    "category_distribution":
        df["Category"].value_counts().to_dict()
        if "Category" in df.columns else {},

    "gender_distribution":
        df["CustomerGender"].value_counts().to_dict()
        if "CustomerGender" in df.columns else {},

    # BUSINESS AGGREGATIONS
    "revenue_by_region":
        df.groupby("Region")["Revenue"]
          .sum()
          .round(2)
          .to_dict()
        if {"Region","Revenue"}.issubset(df.columns)
        else {},

    "revenue_by_category":
        df.groupby("Category")["Revenue"]
          .sum()
          .round(2)
          .to_dict()
        if {"Category","Revenue"}.issubset(df.columns)
        else {},

    # NUMERIC STATISTICS
    "statistics": {
        col: {"mean": round(float(df[col].mean()),2),
            "median": round(float(df[col].median()),2),
            "min": round(float(df[col].min()),2),
            "max": round(float(df[col].max()),2),
            "std": round(float(df[col].std()),2)}
        for col in df.select_dtypes(include=["int64","float64"]).columns},

    # CORRELATIONS
    "correlations":
        df.select_dtypes(include=["int64","float64"]).corr().round(3).to_dict(),

    # TOP PRODUCTS
    "top_products":df["Product"].value_counts().head(10).to_dict()
        if "Product" in df.columns
        else {},

    # DATE RANGE
    "date_range": {
        "start":str(pd.to_datetime(df["OrderDate"],errors="coerce").min()),
        "end":str(pd.to_datetime(df["OrderDate"],errors="coerce").max())} if "OrderDate" in df.columns else {},

    # SAMPLE DATA
    "top_5_records":df.head(5).to_dict(orient="records")}

In [54]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt1 = ChatPromptTemplate.from_messages([
(
"system",
'''
You are a Senior Data Analyst.

Analyze the dataset summary below and generate:

# Dataset Overview

Include:
- What the dataset contains
- Total rows and columns
- Important columns
- Business context inferred from the data

# Data Quality Assessment

Analyze:
- Missing values
- Duplicate rows
- Data type issues
- Data consistency issues
- Potential risks in data quality

Generate:

1. Dataset Overview
2. Important Fields
3. Data Quality Findings
4. Data Quality Score (0-100)
5. Key Data Quality Recommendations

Rules:
- Use only provided data.
- Do not hallucinate.
- Be specific.

Dataset Summary:

{summary}''')])

In [55]:
from langchain_groq import ChatGroq

model = ChatGroq(model="llama-3.1-8b-instant",api_key="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")
chain = prompt1 | model
chain.invoke({"summary": summary})

AIMessage(content="**Dataset Overview**\n\nThe dataset contains information about customer orders, including order details, customer demographics, and sales data. It consists of 1000 rows and 11 columns.\n\n**Important Columns**\n\n1. OrderID: A unique identifier for each order.\n2. OrderDate: The date when the order was placed.\n3. Region: The geographic region where the order was placed.\n4. Category: The product category of the order.\n5. Product: The specific product ordered.\n6. Quantity: The number of products ordered.\n7. UnitPrice: The price of each product.\n8. Revenue: The total revenue generated by the order.\n9. Profit: The profit generated by the order.\n10. CustomerAge: The age of the customer who placed the order.\n11. CustomerGender: The gender of the customer who placed the order.\n\n**Business Context**\n\nThe dataset appears to be from an e-commerce company that sells products in various categories, including electronics, furniture, and clothing. The company has cust

In [56]:
prompt2 = ChatPromptTemplate.from_messages([
(
"system",
'''
You are a Senior Business Intelligence Analyst and Data Scientist.

Analyze the dataset summary below.

Generate:

# Statistical Analysis

Include:
- Important numerical metrics
- Distribution observations
- Outliers and unusual patterns
- High-value variables

# Business Analysis

Analyze:
- Revenue performance
- Profit performance
- Product performance
- Category performance
- Regional performance
- Customer behavior

# Correlation Analysis

Analyze:
- Strong positive correlations
- Strong negative correlations
- Business meaning of each correlation

# Top 10 Data-Driven Insights

Requirements:
- Rank insights by importance.
- Explain why each insight matters.
- Use actual values when available.
- Do not give generic statements.

Rules:
- Use only provided data.
- Never hallucinate.

Dataset Summary:

{summary}''')])

In [57]:
from langchain_groq import ChatGroq

model = ChatGroq(model="llama-3.1-8b-instant",api_key="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")
chain = prompt2 | model
chain.invoke({"summary": summary})

AIMessage(content='# Statistical Analysis\n\n## Important Numerical Metrics\n\n- Mean Revenue: $75,134.96\n- Median Revenue: $29,214.09\n- Mean Profit: $12,652.48\n- Median Profit: $5,724.15\n- Mean Customer Age: 42.66\n- Mean Quantity: 4.49\n- Mean Unit Price: $17,061.58\n\n## Distribution Observations\n\n- Revenue is positively skewed, with a higher number of smaller orders and fewer larger orders.\n- Profit is also positively skewed, indicating that most orders have small profits and fewer orders have substantial profits.\n- Customer Age is relatively normally distributed, with a mean of 42.66 and a standard deviation of 13.72.\n- Quantity is slightly positively skewed, with most orders having a quantity of 1-5 and fewer orders having quantities above 5.\n\n## Outliers and Unusual Patterns\n\n- Revenue outliers: Orders with Revenue above $200,000 (e.g., ORD00005) are considered outliers.\n- Profit outliers: Orders with Profit above $50,000 (e.g., ORD00002) are considered outliers.\n

In [58]:
prompt3 = ChatPromptTemplate.from_messages([
(
"system",
"""
You are an expert Business Intelligence Visualization Architect.

Analyze the dataset summary.

Recommend the 5 most important visualizations.

Return ONLY valid JSON.

Output format:

{{
  "charts":[
    {{
      "chart_type":"",
      "x_column":"",
      "y_column":"",
      "aggregation":"",
      "title":"",
      "business_reason":""
    }}
  ]
}}

Rules:

1. Return ONLY JSON.
2. No explanations.
3. No markdown.
4. No code blocks.
5. Use only columns present in the dataset.
6. Choose the most informative charts.
7. Supported chart types:
   - bar
   - line
   - scatter
   - pie
   - histogram
   - heatmap
   - boxplot

Dataset Summary:

{summary}
"""
)
])

In [59]:
from langchain_groq import ChatGroq

model = ChatGroq(model="llama-3.1-8b-instant",api_key="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")
chain = prompt3 | model
graph_result=chain.invoke({"summary": summary})
graph_result

AIMessage(content='{\n  "charts": [\n    {\n      "chart_type": "bar",\n      "x_column": "Region",\n      "y_column": "Revenue",\n      "aggregation": "sum",\n      "title": "Revenue by Region",\n      "business_reason": "Understand revenue distribution across different regions"\n    },\n    {\n      "chart_type": "bar",\n      "x_column": "Category",\n      "y_column": "Revenue",\n      "aggregation": "sum",\n      "title": "Revenue by Category",\n      "business_reason": "Understand revenue distribution across different categories"\n    },\n    {\n      "chart_type": "pie",\n      "x_column": "Region",\n      "y_column": "Revenue",\n      "aggregation": "sum",\n      "title": "Revenue by Region Distribution",\n      "business_reason": "Get an overview of the revenue distribution across regions"\n    },\n    {\n      "chart_type": "line",\n      "x_column": "OrderDate",\n      "y_column": "Revenue",\n      "aggregation": "sum",\n      "title": "Revenue over Time",\n      "business_re

In [60]:
prompt4 = ChatPromptTemplate.from_messages([
(
"system",
"""
You are a Management Consultant, Strategy Advisor, and Senior Business Analyst.

Using the dataset summary and analytical findings below, generate executive-level recommendations.

Dataset Summary:

{summary}

Analysis Results:

{prompt2}

Generate:

# Executive Summary

Include:
- Biggest opportunity
- Biggest risk
- Most important finding
- Most important recommendation

# Business Recommendations

Generate 5 actionable recommendations.

For each recommendation provide:

1. Recommendation
2. Why it matters
3. Expected business impact
4. Priority (High/Medium/Low)

# Management Action Plan

Generate:

Immediate Actions (0-30 Days)

Short-Term Actions (1-3 Months)

Long-Term Actions (3-12 Months)

Rules:

- Recommendations must be based on findings.
- No generic advice.
- No hallucinations.
- Focus on measurable business impact.
"""
)
])

In [61]:
from langchain_groq import ChatGroq

model = ChatGroq(model="llama-3.1-8b-instant",api_key="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")
chain = prompt4 | model
chain.invoke({"summary": summary,
              "prompt2": prompt2
})

AIMessage(content='**Executive Summary**\n\n**Biggest Opportunity:** The dataset reveals a significant opportunity to increase revenue by targeting the "Electronics" category, which accounted for 55.7% of the total revenue. By focusing on popular products such as Smartphones and Laptops, the company can capitalize on this trend and drive growth.\n\n**Biggest Risk:** The analysis highlights a risk associated with the "Furniture" category, which has seen a decline in revenue by 12.4% compared to the previous year. This trend could continue if not addressed, potentially impacting the company\'s overall revenue.\n\n**Most Important Finding:** The correlation analysis reveals a strong positive relationship between the "Revenue" and "Profit" variables (0.904), indicating that as revenue increases, so does profit. This suggests that the company\'s pricing strategy and product offerings are effective in generating revenue and profit.\n\n**Most Important Recommendation:** Focus on the "Electron

# option 1

In [62]:
prompt = ChatPromptTemplate.from_messages([
(
"system",
"""
You are an expert Python Data Analyst.

Dataset Information:
{dataset_info}

Write ONLY executable pandas code.

Rules:
- Use dataframe name df
- Return only Python code
- No markdown
- No explanations
- Final result must be stored in variable result

Example:

Question:
Highest revenue?

Answer:

result = df["Revenue"].max()

Question:
Average profit?

Answer:

result = df["Profit"].mean()
"""
),
("human","{question}")
])

In [63]:
from langchain_groq import ChatGroq

model = ChatGroq(model="llama-3.1-8b-instant",api_key="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")
chain = prompt | model

In [ ]:
while True:
    question = input("Enter Question: ")
    if question.lower() == "stop":
        break
    response1 = chain.invoke({
        "dataset_info": summary,
        "question": question
    })

    code = response1.content.strip()
    code = code.replace("```python", "")
    code = code.replace("```", "")

    print("\nGenerated Code:")
    print(code)

    local_vars = {"df": df}
    exec(code, {}, local_vars)
    result = local_vars["result"]

    print("\nResult:")
    print(result)

# option 2

In [ ]:
prompt22 = ChatPromptTemplate.from_messages([
(
"system",
"""
You are an expert autonomous data analyst.

Dataset Summary:
{summary}

User Question:
{question}

Instructions:

1. Answer the question directly.
2. Explain the answer using the analysis result.
3. Provide business interpretation.
4. Highlight important observations.
5. Mention any risks or opportunities.
6. Keep the explanation concise and data-driven.
7. Use only the provided data.
8. Do not hallucinate.

Output Format:

Answer:

Explanation:

Business Insight:

Recommendation:
"""
),
("human","{question}")
])

In [ ]:
from langchain_groq import ChatGroq

model = ChatGroq(model="llama-3.1-8b-instant",api_key="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")
chain = prompt22 | model

In [ ]:
while True:
    question = input("Enter Question: ")
    if question.lower() == "stop":
        break
    response = chain.invoke({
        "summary": summary,
        "question": question
    })

    print(response)

In [ ]:
print(graph_result.content)

# chart

In [ ]:
import json

chart_config = json.loads(graph_result.content)

In [ ]:
chart_config

In [ ]:
import matplotlib.pyplot as plt

def generate_chart(df, chart):

    chart_type = chart["chart_type"]
    x = chart["x_column"]
    y = chart["y_column"]

    plt.figure(figsize=(8,5))

    if chart_type == "bar":
        data = (df.groupby(x)[y].sum().sort_values(ascending=False))
        data.plot(kind="bar")

    elif chart_type == "line":
        data = (df.groupby(x)[y].sum())
        data.plot(kind="line")

    elif chart_type == "pie":
        data = (df.groupby(x)[y].sum())
        plt.pie(data.values,labels=data.index,autopct="%1.1f%%")

    elif chart_type == "scatter":
        plt.scatter(df[x],df[y])

    plt.title(chart["title"])
    plt.show()

In [ ]:
for chart in chart_config["charts"]:
    generate_chart(df, chart)